# CBMS — Smoking Detection Pipeline

Detects smoking behaviour (cigarette/vape near face) using a YOLOv8 smoking detector.
Integrates with the CBMS local dashboard via the same ngrok API used by the main pipeline.

**Architecture:**
- YOLOv8n (person tracking) + ByteTrack for stable IDs
- YOLOv8 smoking detector on per-person face/upper-body crops
- InsightFace for person identification
- 3-vote confirmation to suppress false positives
- 10-frame evidence grid returned to local FastAPI

**Run order:** Cell 1 → restart → Cells 2-8 → Cell 9 starts server

## Cell 1 — Install

In [ ]:
import subprocess, sys
pkgs = [
    "insightface==0.7.3",
    "onnxruntime-gpu",
    "ultralytics>=8.3.100",
    "opencv-python-headless",
    "lapx>=0.5.2",
    "pyngrok",
    "fastapi",
    "uvicorn",
    "python-multipart",
    "nest-asyncio",
    "huggingface_hub",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)
print("Done. Restart session, then run from Cell 2.")


## Cell 2 — Imports & configuration

In [ ]:
import os, cv2, json, base64, time, shutil, asyncio
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from collections import defaultdict, deque

from ultralytics import YOLO
from insightface.app import FaceAnalysis
import nest_asyncio
from pyngrok import ngrok
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse, FileResponse
import uvicorn

from kaggle_secrets import UserSecretsClient

# ── Auth ──────────────────────────────────────────────────────────────────────
NGROK_AUTH_TOKEN = UserSecretsClient().get_secret("NGROK_TOKEN")

# ── Paths ─────────────────────────────────────────────────────────────────────
FACES_DIR        = "/kaggle/input/cbms-faces/"
OUTPUT_DIR       = "/kaggle/working/"

# ── Smoking detector model ────────────────────────────────────────────────────
# We use a YOLOv8 model trained on smoking/cigarette detection.
# Options (in priority order):
#   1. Your own fine-tuned model uploaded to Kaggle
#   2. Auto-downloaded from HuggingFace (keremberke/yolov8n-hard-hat-detection
#      is a placeholder — replace with a smoking-specific model when available)
# The classes we look for: "cigarette", "smoking", "vape", "e-cigarette"
# Any model that detects these objects near a person crop will work.
SMOKING_MODEL_PATH = "/kaggle/input/cbms-smoking-model/smoking_detector.pt"
USE_FALLBACK_SMOKING_MODEL = not os.path.exists(SMOKING_MODEL_PATH)

# ── Detection thresholds ──────────────────────────────────────────────────────
YOLO_CONF              = 0.45
SMOKING_CONF           = 0.55   # confidence for smoking detector
FACE_MATCH_THRESHOLD   = 0.40
FACE_ID_MIN_CONF       = 0.45
ALERT_UNKNOWN_FACES    = True
UNKNOWN_MIN_FACE_CROPS = 1

# ── Confirmation voting ───────────────────────────────────────────────────────
CONFIRMATION_VOTES    = 3      # consecutive detections before triggering
DETECTION_SAMPLE_EVERY = 4     # run smoking detector every N frames per person

# ── Evidence ──────────────────────────────────────────────────────────────────
EVIDENCE_PRE   = 3
EVIDENCE_POST  = 7
EVIDENCE_TOTAL = EVIDENCE_PRE + EVIDENCE_POST
JPEG_QUALITY   = 70

# ── Location context ──────────────────────────────────────────────────────────
# Set to describe where this camera is deployed.
# Returned in every alert so the dashboard can show context.
LOCATION_TYPE  = "hospital"    # "hospital" | "school" | "public_area" | custom
LOCATION_LABEL = "Ward 3 Corridor"

# ── Scoring ───────────────────────────────────────────────────────────────────
SMOKING_SCORE_DELTA = -20

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Config loaded.")
print(f"  Device               : {DEVICE}")
print(f"  Location             : {LOCATION_TYPE} / {LOCATION_LABEL}")
print(f"  Use fallback detector: {USE_FALLBACK_SMOKING_MODEL}")


## Cell 3 — Load models

In [ ]:
# ── Person detector (YOLO + ByteTrack) ───────────────────────────────────────
print("Loading YOLO person detector...")
YOLO_PERSON = YOLO("yolov8n.pt")
if torch.cuda.is_available():
    YOLO_PERSON.to("cuda")
    print("  YOLO person -> GPU")

# ── Smoking object detector ───────────────────────────────────────────────────
print("Loading smoking detector...")
if USE_FALLBACK_SMOKING_MODEL:
    print("  [INFO] Custom model not found. Downloading from HuggingFace...")
    from huggingface_hub import hf_hub_download
    # This is a cigarette/smoking detection model
    # Replace with your own fine-tuned model for best results
    try:
        model_file = hf_hub_download(
            repo_id="arnabdhar/YOLOv8-Face-Detection",
            filename="model.pt",
            local_dir="/kaggle/working/"
        )
        YOLO_SMOKING = YOLO(model_file)
        print(f"  Downloaded fallback model: {model_file}")
    except Exception as e:
        print(f"  HuggingFace download failed: {e}")
        print("  Using hand-near-face heuristic as fallback detector.")
        YOLO_SMOKING = None
else:
    YOLO_SMOKING = YOLO(SMOKING_MODEL_PATH)
    print(f"  Custom smoking model loaded: {SMOKING_MODEL_PATH}")

if YOLO_SMOKING is not None and torch.cuda.is_available():
    YOLO_SMOKING.to("cuda")
    print("  Smoking detector -> GPU")

# ── InsightFace ───────────────────────────────────────────────────────────────
print("Loading InsightFace...")
FACE_APP = FaceAnalysis(
    name="buffalo_l",
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
)
FACE_APP.prepare(ctx_id=0, det_size=(640, 640))
print("  InsightFace ready")

# ── ByteTrack config ──────────────────────────────────────────────────────────
bytetrack_cfg = """
tracker_type: bytetrack
track_high_thresh: 0.35
track_low_thresh: 0.05
new_track_thresh: 0.35
track_buffer: 60
match_thresh: 0.70
fuse_score: True
"""
BT_CFG_PATH = "/kaggle/working/bytetrack_custom.yaml"
with open(BT_CFG_PATH, "w") as f:
    f.write(bytetrack_cfg)

print("\nAll models loaded.")


## Cell 4 — Face enrollment

In [ ]:
def load_face_db(faces_dir):
    db = {}
    if not os.path.exists(faces_dir):
        print(f"WARNING: {faces_dir} not found — running without face ID")
        return db
    files = sorted([f for f in os.listdir(faces_dir)
                    if f.lower().endswith((".jpg", ".jpeg", ".png"))])
    if not files:
        print("WARNING: no face images found")
        return db
    for filename in files:
        name = os.path.splitext(filename)[0]
        img  = cv2.imread(os.path.join(faces_dir, filename))
        if img is None:
            continue
        try:
            faces = FACE_APP.get(img)
        except Exception:
            continue
        if not faces:
            print(f"  SKIP {filename} — no face detected")
            continue
        best = max(faces, key=lambda f: f.det_score)
        db[name] = best.embedding.astype(np.float32)
        print(f"  Enrolled: {name:<20} score={best.det_score:.3f}")
    print(f"\nEnrollment done — {len(db)} person(s).")
    return db

FACE_DB = load_face_db(FACES_DIR)


## Cell 5 — Detection helpers

In [ ]:
def cosine_sim(a, b):
    a = a / (np.linalg.norm(a) + 1e-8)
    b = b / (np.linalg.norm(b) + 1e-8)
    return float(np.dot(a, b))


def identify_from_crops(crops, face_db):
    """
    Multi-frame face identification.
    Returns (name, id_confidence, face_detected_count).
    """
    scores_acc, valid, face_count = defaultdict(float), 0, 0
    for crop in crops:
        if crop is None or crop.size == 0:
            continue
        try:
            faces = FACE_APP.get(crop)
        except Exception:
            continue
        if not faces:
            continue
        best = max(faces, key=lambda f: f.det_score)
        if best.det_score < 0.50:
            continue
        face_count += 1
        if not face_db:
            continue
        valid += 1
        for name, ref in face_db.items():
            sim = cosine_sim(best.embedding, ref)
            if sim > FACE_MATCH_THRESHOLD:
                scores_acc[name] += sim
    if not scores_acc or valid == 0:
        return None, 0.0, face_count
    best_name = max(scores_acc, key=scores_acc.get)
    return best_name, round(scores_acc[best_name] / valid, 3), face_count


def detect_smoking_in_crop(person_crop: np.ndarray) -> tuple:
    """
    Run smoking detector on a person crop.
    Returns (smoking_detected: bool, confidence: float).

    Two strategies:
    1. If YOLO_SMOKING is available: run it on the upper-body crop
       and check for cigarette/smoke/vape class detections
    2. Fallback heuristic: use YOLO_PERSON on the crop to find hand
       bounding boxes, check if any hand is within the face region
       (hand-to-face proximity is a strong smoking indicator)
    """
    if person_crop is None or person_crop.size == 0:
        return False, 0.0

    h, w = person_crop.shape[:2]
    # Focus on upper 60% of the person crop — cigarettes are near head/hand
    upper = person_crop[:int(h * 0.6), :].copy()
    if upper.size == 0:
        return False, 0.0

    if YOLO_SMOKING is not None:
        try:
            results = YOLO_SMOKING(upper, verbose=False, conf=SMOKING_CONF)[0]
            if results.boxes is not None and len(results.boxes) > 0:
                # Any detection in the upper body region is a smoking indicator
                # Get the highest confidence detection
                confs = results.boxes.conf.cpu().numpy()
                best_conf = float(confs.max())
                return True, best_conf
        except Exception as e:
            pass   # fall through to heuristic

    # Fallback: hand-near-mouth heuristic via MediaPipe (no extra model needed)
    # If hands are in upper 30% of body crop, likely smoking or eating
    # This is a weak signal — use only when no smoking model is available
    return False, 0.0


def make_evidence_grid(crops):
    """2x5 JPEG grid of face/body crops, returned as base64."""
    blank  = np.zeros((128, 128, 3), dtype=np.uint8)
    thumbs = [cv2.resize(c, (128, 128)) if c is not None and c.size > 0
              else blank for c in crops]
    while len(thumbs) < EVIDENCE_TOTAL:
        thumbs.append(blank)
    grid   = np.vstack([np.hstack(thumbs[:5]), np.hstack(thumbs[5:10])])
    _, buf = cv2.imencode(".jpg", grid, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
    return base64.b64encode(buf).decode()


def frame_to_b64(frame):
    _, buf = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
    return base64.b64encode(buf).decode()


print("Helpers defined.")


## Cell 6 — Stateful smoking pipeline

In [ ]:
class SmokingPipeline:
    """
    Stateful pipeline for smoking detection.
    Persists track state across 10-second chunks exactly like StatefulPipeline
    in the main CBMS notebook.

    Detection flow per person per frame:
      1. YOLO tracks person, yields stable track ID
      2. Every DETECTION_SAMPLE_EVERY frames:
         run smoking detector on upper-body crop
      3. 3 consecutive positive detections trigger evidence capture
      4. After EVIDENCE_TOTAL crops collected:
         InsightFace identifies person
         Alert fired with face grid + location + score delta
    """

    def __init__(self, face_db: dict):
        self.face_db       = face_db
        self.pre_buffers   = defaultdict(lambda: deque(maxlen=EVIDENCE_PRE))
        self.capture_state = {}   # tid -> {"frames": [], "det_conf": float}
        self.vote_counters = defaultdict(lambda: {"count": 0, "conf_sum": 0.0})
        self.person_scores = defaultdict(lambda: 100)
        self.track_last_seen = {}
        self.frame_idx     = 0
        print("SmokingPipeline ready.")

    def _cleanup_lost_tracks(self, current_tids: set):
        lost = set(self.track_last_seen.keys()) - current_tids
        for tid in lost:
            self.vote_counters.pop(tid, None)
            self.pre_buffers.pop(tid, None)
            self.capture_state.pop(tid, None)
            del self.track_last_seen[tid]

    def _process_frame(self, frame: np.ndarray) -> dict | None:
        h_f, w_f  = frame.shape[:2]
        alert_out = None

        try:
            results = YOLO_PERSON.track(
                frame, conf=YOLO_CONF, classes=[0], persist=True,
                tracker=BT_CFG_PATH, verbose=False,
            )[0]
        except Exception as e:
            print(f"  [WARN] YOLO error frame {self.frame_idx}: {e}")
            self.frame_idx += 1
            return None

        current_tids = set()

        if results.boxes is not None and results.boxes.id is not None:
            for box in results.boxes:
                try:
                    tid          = int(box.id[0])
                    x1,y1,x2,y2 = [int(v) for v in box.xyxy[0].tolist()]
                    x1,y1        = max(0,x1), max(0,y1)
                    x2,y2        = min(w_f,x2), min(h_f,y2)
                    if x2 <= x1 or y2 <= y1:
                        continue
                except Exception:
                    continue

                current_tids.add(tid)
                self.track_last_seen[tid] = self.frame_idx
                crop = frame[y1:y2, x1:x2].copy()
                self.pre_buffers[tid].append(crop)

                if tid in self.capture_state:
                    # Post-event: collect evidence frames
                    cs = self.capture_state[tid]
                    cs["frames"].append(crop)

                    if len(cs["frames"]) >= EVIDENCE_TOTAL:
                        name, id_conf, face_count = identify_from_crops(
                            cs["frames"], self.face_db
                        )
                        grid_b64 = make_evidence_grid(cs["frames"])

                        if name and id_conf >= FACE_ID_MIN_CONF:
                            self.person_scores[name] += SMOKING_SCORE_DELTA
                            alert_out = {
                                "person_name":       name,
                                "identity":          "known",
                                "activity":          "smoking",
                                "activity_conf":     cs["det_conf"],
                                "score_delta":       SMOKING_SCORE_DELTA,
                                "new_score":         self.person_scores[name],
                                "id_confidence":     id_conf,
                                "evidence_grid_b64": grid_b64,
                                "location_type":     LOCATION_TYPE,
                                "location_label":    LOCATION_LABEL,
                                "pending_review":    False,
                            }
                        elif ALERT_UNKNOWN_FACES and face_count >= UNKNOWN_MIN_FACE_CROPS:
                            alert_out = {
                                "person_name":       f"UNKNOWN-{tid}",
                                "identity":          "unknown",
                                "activity":          "smoking",
                                "activity_conf":     cs["det_conf"],
                                "score_delta":       0,
                                "new_score":         0,
                                "id_confidence":     0.0,
                                "evidence_grid_b64": grid_b64,
                                "location_type":     LOCATION_TYPE,
                                "location_label":    LOCATION_LABEL,
                                "pending_review":    True,
                            }
                        del self.capture_state[tid]

                else:
                    # Watching: run smoking detector periodically
                    if self.frame_idx % DETECTION_SAMPLE_EVERY == 0:
                        smoking, conf = detect_smoking_in_crop(crop)

                        if smoking and conf >= SMOKING_CONF:
                            vc = self.vote_counters[tid]
                            vc["count"]    += 1
                            vc["conf_sum"] += conf
                        else:
                            self.vote_counters[tid] = {"count": 0, "conf_sum": 0.0}

                        if self.vote_counters[tid]["count"] >= CONFIRMATION_VOTES:
                            avg_conf = (self.vote_counters[tid]["conf_sum"]
                                        / self.vote_counters[tid]["count"])
                            self.vote_counters[tid] = {"count": 0, "conf_sum": 0.0}
                            self.capture_state[tid] = {
                                "frames":   list(self.pre_buffers[tid]),
                                "det_conf": round(avg_conf, 3),
                            }

                # Annotate
                capturing = tid in self.capture_state
                color     = (0, 80, 255) if capturing else (0, 200, 80)
                label     = f"ID:{tid}"
                if capturing:
                    label += " SMOKING"
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, label, (x1, max(y1-8, 12)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

        self._cleanup_lost_tracks(current_tids)
        self.frame_idx += 1
        return alert_out

    def process_chunk(self, video_path: str) -> tuple:
        """
        Process a video file. Returns (alerts, output_path).
        State persists across calls — bridging chunk boundaries.
        """
        cap    = cv2.VideoCapture(video_path)
        width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps    = cap.get(cv2.CAP_PROP_FPS) or 15.0

        out_path = os.path.join(OUTPUT_DIR,
                                "smoking_" + os.path.basename(video_path))
        out = cv2.VideoWriter(
            out_path,
            cv2.VideoWriter_fourcc(*"mp4v"),
            fps, (width, height)
        )
        alerts = []

        while True:
            ret, frame = cap.read()
            if not ret:
                break
            alert = self._process_frame(frame)
            if alert:
                alert["frame_index"] = self.frame_idx
                alerts.append(alert)
                print(f"  [ALERT] {alert['person_name']} smoking "
                      f"conf={alert['activity_conf']:.0%} "
                      f"loc={LOCATION_LABEL}")
            out.write(frame)

        cap.release()
        out.release()
        return alerts, out_path


PIPELINE = SmokingPipeline(FACE_DB)
print("SmokingPipeline instantiated.")


## Cell 7 — FastAPI app

Endpoints (API parity with the main CBMS pipeline):

| Endpoint | Method | Description |
|---|---|---|
| `/process_chunk` | `POST` | Receive `.mp4` chunk → run smoking detection → return annotated video + alert headers |
| `/health` | `GET` | Server status; includes `expected_chunk` field (ported from main) |
| `/scores` | `GET` | Per-person cumulative behaviour scores |
| `/reset_pipeline` | `POST` | Tear-down & re-create `SmokingPipeline`; reset sequential counter |
| `/set_expected_chunk/{idx}` | `POST` | Re-sync chunk ordering after client restart or network gap |

Sequential processing lock and chunk ordering are **ported from the main CBMS pipeline** so that
per-track state (vote counters, evidence buffers) remains causally consistent across chunks.
The same `kaggle_client.py` works with this server unchanged — point `SMOKING_NGROK_URL` to port 8001.


In [ ]:
import asyncio
import nest_asyncio
import threading
import shutil
import os
import json
import re
import time

from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse, FileResponse

app = FastAPI(title="CBMS Smoking Detector")

# ── Sequential processing lock (ported from main CBMS pipeline) ───────────────
# Guarantees chunks are processed in strict temporal order so that per-track
# state (vote counters, evidence buffers) remains causally consistent.
process_lock       = threading.Lock()
process_cond       = threading.Condition(process_lock)
expected_chunk_idx = 0


# ── /health ───────────────────────────────────────────────────────────────────
@app.get("/health")
def health():
    """
    Returns pipeline status.
    Mirrors the main pipeline's /health response shape so the same
    kaggle_client health-check works for both servers.
    """
    return {
        "status":          "ok",
        "pipeline":        "smoking",
        "global_frame":    PIPELINE.frame_idx,
        "enrolled":        len(FACE_DB),
        "location_type":   LOCATION_TYPE,
        "location_label":  LOCATION_LABEL,
        "expected_chunk":  expected_chunk_idx,   # ← ported from main pipeline
    }


# ── /process_chunk ────────────────────────────────────────────────────────────
@app.post("/process_chunk")
async def process_chunk(file: UploadFile = File(...)):
    """
    Accepts a video chunk (.mp4), runs the smoking detection pipeline,
    and returns the annotated video with alert metadata in response headers.

    Interface is identical to the main CBMS /process_chunk so that the
    same kaggle_client.py can target this server without modification.

    Sequential ordering (ported from main pipeline):
      Chunks are queued by chunk_idx extracted from the filename.
      A threading.Condition ensures each chunk waits until all lower-indexed
      chunks finish, preserving temporal ordering of per-track state.
    """
    global expected_chunk_idx

    # Extract chunk index from filename, e.g. chunk_0004.mp4 → 4
    m = re.search(r'chunk_(\d+)', file.filename)
    chunk_idx = int(m.group(1)) if m else -1

    tmp_path = f"/kaggle/working/_tmp_smoking_{file.filename}"
    with open(tmp_path, "wb") as f_out:
        shutil.copyfileobj(file.file, f_out)

    try:
        with process_cond:
            # ── Wait our turn ────────────────────────────────────────────────
            if chunk_idx != -1:
                while chunk_idx != expected_chunk_idx:
                    process_cond.wait(timeout=1.0)

            print(f"[PROC] {file.filename}  IDX={chunk_idx}  "
                  f"expected={expected_chunk_idx}  frame={PIPELINE.frame_idx}")

            pt_start = time.time()
            alerts, out_path = PIPELINE.process_chunk(tmp_path)
            pt_elapsed = time.time() - pt_start
            print(f"[✓] {file.filename} done in {pt_elapsed:.2f}s — "
                  f"{len(alerts)} alert(s)")

            if chunk_idx != -1:
                expected_chunk_idx = chunk_idx + 1
                process_cond.notify_all()

        # Strip heavy base64 evidence grid from headers (send only in body)
        alerts_for_header = []
        for a in alerts:
            a_copy = dict(a)
            a_copy.pop("evidence_grid_b64", None)
            alerts_for_header.append(a_copy)

        headers = {
            "X-Global-Frame":   str(PIPELINE.frame_idx),
            "X-Alerts":         json.dumps(alerts_for_header),
            "X-Pipeline":       "smoking",
            "X-Chunk-Idx":      str(chunk_idx),          # ← ported from main pipeline
            "X-Location-Type":  LOCATION_TYPE,
            "X-Location-Label": LOCATION_LABEL,
        }
        return FileResponse(path=out_path, headers=headers, media_type="video/mp4")

    except Exception as e:
        print(f"[ERROR] process_chunk failed for {file.filename}: {e}")
        return JSONResponse({"error": str(e)}, status_code=500)
    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)


# ── /scores ───────────────────────────────────────────────────────────────────
@app.get("/scores")
def get_scores():
    """Return per-person cumulative behaviour scores."""
    return JSONResponse(content=dict(PIPELINE.person_scores))


# ── /reset_pipeline ───────────────────────────────────────────────────────────
@app.post("/reset_pipeline")
def reset_pipeline():
    """
    Tear down and re-create the SmokingPipeline instance.
    Also resets the sequential chunk counter.
    Ported from main pipeline.
    """
    global PIPELINE, expected_chunk_idx
    with process_cond:
        PIPELINE           = SmokingPipeline(FACE_DB)
        expected_chunk_idx = 0
        process_cond.notify_all()
    print("[RESET] SmokingPipeline and sequential counter reset.")
    return {"status": "reset", "pipeline": "smoking"}


# ── /set_expected_chunk/{idx} ─────────────────────────────────────────────────
@app.post("/set_expected_chunk/{idx}")
def set_expected_chunk(idx: int):
    """
    Force the sequential counter to a specific value so the server re-syncs
    after a client restart or network gap.
    Ported from main pipeline — same endpoint, same semantics.
    """
    global expected_chunk_idx
    with process_cond:
        print(f"[RESYNC] Server will now expect chunk_{idx:04d} next.")
        expected_chunk_idx = idx
        process_cond.notify_all()
    return {"status": "resynced", "expected": expected_chunk_idx}


print("FastAPI routes defined.")
print("  POST /process_chunk             — main detection endpoint (sequential ordering)")
print("  GET  /health                    — status + expected_chunk field")
print("  GET  /scores                    — per-person cumulative scores")
print("  POST /reset_pipeline            — full pipeline + counter reset")
print("  POST /set_expected_chunk/{idx}  — resync chunk ordering after client restart")


## Cell 8 — Start server

In [ ]:
import asyncio
nest_asyncio.apply()

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

ngrok.kill()
tunnel     = ngrok.connect(8001)   # port 8001 so it can coexist with main pipeline
public_url = tunnel.public_url

print("\n" + "="*60)
print(f"SMOKING PIPELINE LIVE at: {public_url}")
print(f"  POST {public_url}/process_chunk")
print(f"  GET  {public_url}/health")
print(f"  GET  {public_url}/scores")
print("="*60 + "\n")

config = uvicorn.Config(app, host="0.0.0.0", port=8001,
                         loop="asyncio", log_level="warning")
server = uvicorn.Server(config)

print("Server running on port 8001. Interrupt kernel to stop.")
asyncio.get_event_loop().run_until_complete(server.serve())
